# Step 0 — Build Database

Reads the three source CSVs, builds the SQLite database, and populates the `persons` table with one row per researchable individual.

`names_qualified_papers.csv` uses `person_id` to group names that appeared together in a single original newspaper entry. Because multiple distinct people can be listed under one entry (e.g., "Smith & Jones, editors"), each name gets its own `research_id` in the `persons` table — but they share a `person_id`.

> **Note:** The `name_variants` field is reserved for genuine alternate spellings
> (e.g. "Whitelaw Reid" / "Whitlaw Reid"). Co-listed editors sharing a `person_id`
> are *not* variants — they are different people.

In [1]:
import sys, json
from pathlib import Path
import pandas as pd
sys.path.insert(0, str(Path('.')))
from db import init_db, get_connection, DB_PATH

DATA = Path('../data/personnel_coding')
MASTER = Path('../data/master.csv')

In [2]:
conn = init_db()

Database initialised at c:\Users\samwt\Documents\ExtendedEssay\owner_bio_scraping\railroad_pipeline.db


## Build persons table

In [3]:
nqp = pd.read_csv(DATA / 'names_qualified_papers.csv')
oe  = pd.read_csv(DATA / 'owners_and_editors.csv')
master = pd.read_csv(MASTER, low_memory=False)

print(f"names_qualified_papers: {len(nqp)} rows, {nqp['person_id'].nunique()} unique person_ids")
print(f"owners_and_editors: {len(oe)} rows, {oe['person_id'].notna().sum()} with person_id")

names_qualified_papers: 570 rows, 463 unique person_ids
owners_and_editors: 2088 rows, 808 with person_id


In [4]:
# Build per-person_id metadata from owners_and_editors
pid_meta = {}
for pid, grp in oe[oe['person_id'].notna()].groupby('person_id'):
    pid = int(pid)
    states = sorted(grp['state'].dropna().str.strip().unique().tolist())
    newspapers = sorted(grp['newspaper_name'].dropna().str.strip().unique().tolist())
    first_yr = int(grp['first_year'].min()) if grp['first_year'].notna().any() else None
    last_yr  = int(grp['last_year'].max())  if grp['last_year'].notna().any()  else None
    pid_meta[pid] = {
        'states': states,
        'newspapers': newspapers,
        'first_year': first_yr,
        'last_year': last_yr,
    }
print(f"Built metadata for {len(pid_meta)} person_ids")

Built metadata for 463 person_ids


In [ ]:
# Check if already populated
existing = conn.execute("SELECT COUNT(*) as n FROM persons").fetchone()["n"]
if existing > 0:
    print(f"persons table already has {existing} rows — skipping insert. Delete the DB to rebuild.")
else:
    COMMON_SURNAMES = {'smith', 'jones', 'brown', 'johnson', 'williams', 'davis', 'miller',
                       'wilson', 'moore', 'taylor', 'anderson', 'thomas', 'jackson', 'white',
                       'harris', 'martin', 'thompson', 'garcia', 'martinez', 'robinson'}

    rows_inserted = 0
    for _, row in nqp.iterrows():
        pid  = int(row['person_id'])
        name = str(row['name']).strip()
        meta = pid_meta.get(pid, {})

        # name_variants is reserved for genuine alternate spellings,
        # NOT co-listed editors sharing a person_id.
        # Leave NULL; populate manually or via enrichment if needed.
        
        # Ambiguity heuristic
        parts = name.split()
        surname = parts[-1].lower() if parts else ''
        is_ambiguous = 1 if len(parts) <= 1 or surname in COMMON_SURNAMES else 0

        conn.execute(
            """INSERT INTO persons
               (person_id, canonical_name, name_variants, known_states, known_newspapers,
                first_active_year, last_active_year, is_ambiguous)
               VALUES (?, ?, ?, ?, ?, ?, ?, ?)""",
            (pid, name,
             None,  # no variants — co-editors are different people
             json.dumps(meta.get('states', [])),
             json.dumps(meta.get('newspapers', [])),
             meta.get('first_year'), meta.get('last_year'),
             is_ambiguous)
        )
        rows_inserted += 1

    conn.commit()
    print(f"Inserted {rows_inserted} persons")
    ambig = conn.execute("SELECT COUNT(*) as n FROM persons WHERE is_ambiguous=1").fetchone()["n"]
    print(f"  {ambig} flagged as potentially ambiguous")

## Initialize pipeline_progress rows for all persons

In [6]:
STEPS = [
    '1_wikidata',
    '2a_google_books',
    '2b_hathitrust',
    '2c_dpla',
    '3a_internet_archive',
    '3b_chronicling_america',
    '4_text_assembly',
    '5_extraction',
    '6_feedback_loop',
]

persons = conn.execute("SELECT research_id FROM persons").fetchall()
inserted = 0
for p in persons:
    for step in STEPS:
        try:
            conn.execute(
                "INSERT OR IGNORE INTO pipeline_progress (research_id, step, status) VALUES (?, ?, 'pending')",
                (p["research_id"], step)
            )
            inserted += 1
        except Exception:
            pass
conn.commit()
print(f"Created {inserted} progress tracking rows")

Created 5130 progress tracking rows


## Summary

In [7]:
from db import pipeline_summary
print("=== Database Summary ===")
print(f"DB path: {DB_PATH}")
print()
total = conn.execute("SELECT COUNT(*) as n FROM persons").fetchone()["n"]
ambig = conn.execute("SELECT COUNT(*) as n FROM persons WHERE is_ambiguous=1").fetchone()["n"]
print(f"Persons:    {total}")
print(f"  Ambiguous: {ambig}")
print()
print("Sample persons:")
for row in conn.execute("SELECT research_id, canonical_name, known_states, first_active_year, last_active_year FROM persons LIMIT 15").fetchall():
    states = json.loads(row['known_states'] or '[]')
    print(f"  [{row['research_id']:4d}] {row['canonical_name']:<35} {', '.join(states[:2]):<20} {row['first_active_year'] or '?'}\u2013{row['last_active_year'] or '?'}")

=== Database Summary ===
DB path: c:\Users\samwt\Documents\ExtendedEssay\owner_bio_scraping\railroad_pipeline.db

Persons:    570
  Ambiguous: 37

Sample persons:
  [   1] Whitelaw Reid                       NEW YORK             1873–1890
  [   2] Charles A. Dana                     NEW YORK             1869–1871
  [   3] George Ripley                       NEW YORK             1869–1871
  [   4] Bayard Taylor                       NEW YORK             1869–1871
  [   5] Henry J. Raymond                    NEW YORK             1869–1871
  [   6] Horace Greeley                      NEW YORK             1869–1871
  [   7] Margaret Fuller                     NEW YORK             1869–1871
  [   8] Crosby Noyes                        DISTRICT OF COLUMBIA 1869–1890
  [   9] Edgar Snowden                       VIRGINIA             1869–1885
  [  10] William C. Dunbar                   UTAH                 1890–1890
  [  11] James Gordon Bennett                NEW YORK             1871–1885
 